# Predicting food inflation in Turkey

**Question:** Do exchange rates and global food prices help predict monthly food inflation in Turkey?

We will compare Turkish food-price changes with USD/TRY changes and global food-price changes, including related food groups.

The cell below downloads three CSV files into `data/raw/`: `turkish_food_cpi.csv`, `usd_try_daily.csv` and `fao_food_prices.csv`. Turkish CPI and USD/TRY are requested from January 2005 through the last completed month. Yearly EVDS responses are combined before saving each source. Each source's original column names and date formats are retained. No monthly aggregation, gap filling or merging between sources is done.

- **Turkish food CPI and food categories:** [TÜİK](https://veriportali.tuik.gov.tr/tr/press/58289), downloaded through [TCMB EVDS](https://evds3.tcmb.gov.tr/). CPI is a price index; 2025=100 is its reference scale.
- **USD/TRY exchange rate:** [TCMB EVDS](https://evds3.tcmb.gov.tr/), daily buying rates.
- **Global food prices:** [FAO Food Price Index](https://www.fao.org/worldfoodsituation/foodpricesindex/en/), monthly overall and food-group indices. The CSV includes its full available history.

Install the libraries listed in `requirements.txt` in the notebook's Python environment with `python -m pip install -r requirements.txt`. To get the free API key, open [EVDS](https://evds3.tcmb.gov.tr/), choose “Benim Sayfam / My Page”, register if needed, and sign in. Open your username menu, choose “Profilim / Profile”, and select “API Anahtarını Kopyala / Copy API Key” near the bottom. Paste the key after `EVDS_API_KEY=` in the local `.env` file and save it. The cell reads that file automatically and uses a hidden prompt if the key is still empty. The `.env` file is excluded from Git. Do not paste the key into the notebook code or this chat. See the [official API guide](https://evds3.tcmb.gov.tr/igmevdsms-dis/documents/showDocument?docId=16). The EVDS requests are split by year to stay below the API's 1,000-observation limit. FAO's title rows and unused trailing columns are omitted from the saved CSV.

TCMB's [FAQ](https://www.tcmb.gov.tr/wps/wcm/connect/bab69efb-d66c-45b5-91c6-1533886acd6e/GenelAg-SSS.pdf?CACHE=NONE&CONTENTCACHE=NONE&MOD=AJPERES&attachment=true) describes EVDS exchange rates as rates determined on the previous business day. Our spot checks match that convention. We retain the EVDS dates rather than shifting them.


In [ ]:
import io
import os
from datetime import date, timedelta
from getpass import getpass
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)
end_date = date.today().replace(day=1) - timedelta(days=1)
load_dotenv(".env")
api_key = os.getenv("EVDS_API_KEY", "").strip() or getpass("EVDS API key: ").strip()
if not api_key:
    raise ValueError("Enter your EVDS API key in .env or the hidden prompt.")

fao_url = (
    "https://www.fao.org/media/docs/worldfoodsituationlibraries/wfs-library/food_price_indices_data.csv?download=true&sfvrsn=523ebd2a_84"
)
response = requests.get(fao_url, timeout=90)
response.raise_for_status()
fao = pd.read_csv(io.BytesIO(response.content), skiprows=2, usecols=range(7)).dropna(how="all")
fao.to_csv(raw_dir / "fao_food_prices.csv", index=False)
print(f"Saved fao_food_prices.csv: {len(fao)} months")

series = {
    "turkish_food_cpi": [
        "TP.TUKFIY2025.01",
        "TP.TUKFIY2025.0111",
        "TP.TUKFIY2025.0112",
        "TP.TUKFIY2025.0114",
        "TP.TUKFIY2025.0115",
    ],
    "usd_try_daily": ["TP.DK.USD.A.YTL"],
}
for dataset, codes in series.items():
    yearly_tables = []
    for year in range(2005, end_date.year + 1):
        year_end = min(date(year, 12, 31), end_date)
        url = (
            f"https://evds3.tcmb.gov.tr/igmevdsms-dis/series={'-'.join(codes)}&startDate=01-01-{year}&endDate={year_end:%d-%m-%Y}&type=csv"
        )
        response = requests.get(url, headers={"key": api_key}, timeout=90)
        response.raise_for_status()
        yearly_tables.append(pd.read_csv(io.BytesIO(response.content)))

    data = pd.concat(yearly_tables, ignore_index=True)
    data.to_csv(raw_dir / f"{dataset}.csv", index=False)
    print(f"Saved {dataset}.csv: {len(data)} observations")
